# Pré-processamento

**OBS.:** Assumimos que a análise exploratória de dados (EDA) já foi feita.

Esse exemplo vai cobrir
+ Divisão do dataset
+ Escalonamento de atributos
+ Codificação de atributos
+ Engenharia de atributos
+ Extração de atributos
+ Seleção de atributos

In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. Baixando o dataset

Os dados referem-se a campanhas de marketing direto (chamadas telefônicas) de uma instituição bancária portuguesa.

O objetivo é classificar se um cliente contratará um depósito a prazo ou não a partir de 16 atributos.

Portanto, este é um problema de classificação binária com 16 atributos (variável y é binária: cliente contrata ou não o depósito).

**OBS.:** Temos atributos numéricos e categóricos.

In [48]:
url = "https://raw.githubusercontent.com/zz4fap/c24_inteligencia_artificial/main/data/bank_marketingv1.csv"
df = pd.read_csv(url)

# Mostrando as 5 primeiras linhas
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,33,admin.,married,tertiary,no,882,no,no,telephone,21,oct,39,1,151,3,failure,0
1,42,admin.,single,secondary,no,-247,yes,yes,telephone,21,oct,519,1,166,1,other,1
2,33,services,married,secondary,no,3444,yes,no,telephone,21,oct,144,1,91,4,failure,1
3,36,management,married,tertiary,no,2415,yes,no,telephone,22,oct,73,1,86,4,other,0
4,36,management,married,tertiary,no,0,yes,no,telephone,23,oct,140,1,143,3,failure,1


### Valores únicos das variáveis categóricas

In [49]:
categorical_features = df.select_dtypes(
    include=["object", "category"]
).columns

df[categorical_features].nunique()

,0
job,11
marital,3
education,3
default,2
housing,2
loan,2
contact,2
month,12
poutcome,3


## 2. Separação das colunas entre atributos (X) e rótulos (y).

In [50]:
# Matriz de atributos
X = df.drop(columns=["y"])

# Vetor de rótulos
y = df["y"]

**OBS**.: O rótulo identifica dois grupos, ou classes, classe de quem contratou, `yes=1` e a classe de quem não contratou, `no=0`.

### Tamanho original do dataset

In [51]:
X.shape

(7842, 16)

## 3. Divisão do dataset

`test_size` define o tamanho do subconjunto de teste.

O código abaixo divide o conjunto original em 70% para treinamento e 30% para validação e teste.

In [52]:
X_train, X_val_test, y_train, y_val_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

O código abaixo divide o conjunto contendo amostras de validação e teste em dois conjuntos um com 50% para validação e outro com 50% para teste.

In [53]:
X_val, X_test, y_val, y_test = train_test_split(
    X_val_test, y_val_test, test_size=0.5, random_state=42
)

Ao final, temos os conjuntos de treinamento com 70%, e de validação e teste com 15% cada.

### Dimensões dos conjuntos de treinamento, validação e teste.

In [54]:
X_train.shape

(5489, 16)

In [55]:
X_val.shape

(1176, 16)

In [56]:
X_test.shape

(1177, 16)

## 4. Engenharia de atributos (exemplo simples)

Vamos criar 2 novos atributos.

#### Novo atributo: Intensidade da campanha.

A ideia é medir quantos contatos estão sendo feitos agora em relação ao histórico anterior.

O +1 evita divisão por zero.

In [57]:
def intensidade_da_campanha(X):
    X = X.copy()
    X["campaign_intensity"] = X["campaign"] / (X["previous"] + 1)
    return X

#### Novo atributo: Número total de contatos

O dataset possui campaign = número de contatos durante a campanha atual e previous = número de contatos anteriores.

In [58]:
def total_de_contatos(X):
    X = X.copy()
    X["total_contacts"] = X["campaign"] + X["previous"]
    return X

#### Criar dois novos atributos

In [59]:
# Cria os dois atributos em sequência.
def feat_eng(X):
  X = intensidade_da_campanha(X)
  return total_de_contatos(X)

# Transforma a função acima em um "transformador" de dados que pode ser usado com outras classes da biblioteca SciKit-Learn.
feature_engineering = FunctionTransformer(feat_eng)

## 5. Escalonamento e codificação dos atributos

Usamos a classe `ColumnTransformer`, que permite aplicar transformações diferentes a colunas diferentes, tudo em um único objeto.

In [60]:
numeric_features = ['age', 'balance', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'campaign_intensity', 'total_contacts']
ordinal_features = ['month', 'education']
nominal_features = ['job', 'marital', 'contact', 'poutcome']
binary_features = ['default', 'housing', 'loan']

# O ColumnTransformer aplica transformações por coluna especificada.
preprocessing = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("ordinal", OrdinalEncoder(), ordinal_features),
        ("nominal", OneHotEncoder(), nominal_features),
        ("binary", OrdinalEncoder(), binary_features)
    ]
)

## 6. Seleção e extração de atributos

+ Seleção: remove atributos com baixa variância, no caso, menor do que o limiar especificado.

+ Extração: PCA (seleciona a quantidade de componentes principais que mantém 95% da variância explicada)

In [61]:
feature_selection = VarianceThreshold(threshold=0.01)
feature_extraction = PCA(n_components=0.95)

## 7. Pipeline completo

**OBS.1**.: Usamos um modelo de classificação binária, o `Regressor logístico`.

**OBS.2**: Com a classe `Pipeline`, podemos combinar todas as etapas acima.

**OBS.3**: Fazemos a seleção de atributos antes da extração para eliminar atributos irrelevantes, ruidosos e altamente correlacionados antes do PCA, que é sensível a esses problemas.


In [62]:
pipeline = Pipeline(steps=[
    ("feature_engineering", feature_engineering),
    ("preprocessing", preprocessing),
    ("feature_selection", feature_selection),
    ("feature_extraction", feature_extraction),
    ("model", LogisticRegression())
])

## 8. Treinamento do modelo de ML

In [63]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function feat_eng at 0x789c36df5ee0>)),
                ('preprocessing',
                 ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                                  ['age', 'balance',
                                                   'day_of_week', 'duration',
                                                   'campaign', 'pdays',
                                                   'previous',
                                                   'campaign_intensity',
                                                   'total_contacts']),
                                                 ('ordinal', OrdinalEncoder(),
                                                  ['month', 'education']),
                                                 ('nominal', OneHotEncoder(),
                                                  ['job', 'marital', 'contact',
                                                   'poutcome']),
                                                 ('binary', OrdinalEncoder(),
                                                  ['default', 'housing',
                                                   'loan'])])),
                ('feature_selection', VarianceThreshold(threshold=0.01)),
                ('feature_extraction', PCA(n_components=0.95)),
                ('model', LogisticRegression())])

## 9. Desempenho do modelo no conjunto de treinamento

In [64]:
# Usamos o modelo treinado para realizar inferências.
y_train_pred = pipeline.predict(X_train)

# Cálculo da acurácia do conjunto de treinamento.
train_accuracy = accuracy_score(y_train, y_train_pred)

print(f"Acurácia no conjunto de treinamento: {train_accuracy:.2f}")

Acurácia no conjunto de treinamento: 0.83


## 10. Desempenho do modelo no conjunto de validação

In [65]:
# Usamos o modelo treinado para realizar inferências.
y_val_pred = pipeline.predict(X_val)

# Cálculo da acurácia do conjunto de validação.
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Acurácia no conjunto de validação: {val_accuracy:.2f}")

Acurácia no conjunto de validação: 0.82


## 11. Desempenho do modelo no conjunto de teste

In [66]:
# Usamos o modelo treinado para realizar inferências.
y_test_pred = pipeline.predict(X_test)

# Cálculo da acurácia do conjunto de teste.
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Acurácia no conjunto de teste: {test_accuracy:.2f}")

Acurácia no conjunto de teste: 0.84


## 12. Número de componentes principais usadas pelo PCA

In [67]:
pipeline['feature_extraction'].n_components_

np.int64(14)

### 13. Nomes dos atributos após a codificação

In [68]:
# Obtém os nomes dos atributos após o preprocessing
colunas = pipeline['preprocessing'].get_feature_names_out()

print("Quantidade de atributos após o preprocessing:")
print(len(colunas))
print("\nAtributos após o preprocessing:")
for coluna in colunas:
    print(coluna)

Quantidade de atributos após o preprocessing:
33

Atributos após o preprocessing:
numeric__age
numeric__balance
numeric__day_of_week
numeric__duration
numeric__campaign
numeric__pdays
numeric__previous
numeric__campaign_intensity
numeric__total_contacts
ordinal__month
ordinal__education
nominal__job_admin.
nominal__job_blue-collar
nominal__job_entrepreneur
nominal__job_housemaid
nominal__job_management
nominal__job_retired
nominal__job_self-employed
nominal__job_services
nominal__job_student
nominal__job_technician
nominal__job_unemployed
nominal__marital_divorced
nominal__marital_married
nominal__marital_single
nominal__contact_cellular
nominal__contact_telephone
nominal__poutcome_failure
nominal__poutcome_other
nominal__poutcome_success
binary__default
binary__housing
binary__loan


### 14. Atributos eliminados pelo VarianceThreshold

In [69]:
# Obtém a máscara dos atributos selecionados pelo VarianceThreshold
support = pipeline['feature_selection'].get_support()

# Atributos eliminados
atributos_eliminados = colunas[~support]

print("Atributos eliminados pelo VarianceThreshold:")
for atributo in atributos_eliminados:
    print(f"- {atributo}")

Atributos eliminados pelo VarianceThreshold:
- binary__default
